# U1-01 · Python, NumPy, SciPy, NetworkX y Matplotlib

*Programación de Redes Neuronales - Machine Learning (IFCD48) · Unidad 1, apartados 1.1 y 1.2*

Ejecuta cada celda con **Mayús + Intro**. Todo funciona en el navegador, sin instalar nada.

## 1. Pensar en vectores

In [ ]:
import time
import numpy as np

n = 1_000_000
lista = list(range(n))
vector = np.arange(n, dtype=np.float64)

t0 = time.perf_counter()
total = 0.0
for x in lista:                 # bucle de Python: elemento a elemento
    total += x * x
t_bucle = time.perf_counter() - t0

t0 = time.perf_counter()
total_np = np.dot(vector, vector)  # una sola operación vectorial
t_numpy = time.perf_counter() - t0

print(f"bucle: {t_bucle * 1000:.1f} ms, "
      f"NumPy: {t_numpy * 1000:.2f} ms")
print(f"{t_bucle / t_numpy:.0f} veces más rápido")
print(f"iguales: {total == total_np}; diferencia relativa: "
      f"{abs(total - total_np) / total:.1e}")
print("memoria:", vector.nbytes // 1024, "KB en float64,",
      vector.astype(np.float32).nbytes // 1024, "KB en float32")

**Prueba.** Cambia `n` a 10 millones. ¿Crece la ventaja de NumPy?

## 2. Aprender es optimizar

In [ ]:
import numpy as np
from scipy.optimize import minimize

rng = np.random.default_rng(0)
x = rng.uniform(0, 10, 50)
y = 2 * x + 1 + rng.normal(0, 1, 50)        # recta real: 2x + 1

def error(p):                               # error cuadrático medio
    w, b = p
    return np.mean((w * x + b - y) ** 2)

# 1) un optimizador de SciPy busca los parámetros
r = minimize(error, x0=[0.0, 0.0])
print(f"SciPy: w = {r.x[0]:.3f}, b = {r.x[1]:.3f}, "
      f"error = {r.fun:.3f}, {r.nit} iteraciones")

# 2) lo mismo, a mano, con descenso de gradiente
w, b, eta = 0.0, 0.0, 0.01
for paso in range(5000):
    e = w * x + b - y
    w -= eta * 2 * np.mean(e * x)           # derivada respecto a w
    b -= eta * 2 * np.mean(e)               # derivada respecto a b
print(f"gradiente: w = {w:.3f}, b = {b:.3f}, "
      f"error = {error([w, b]):.3f}")

# 3) solución exacta por mínimos cuadrados
A = np.column_stack([x, np.ones_like(x)])
print("exacta:   w = {:.3f}, b = {:.3f}".format(
    *np.linalg.lstsq(A, y, rcond=None)[0]))

**Prueba.** En el descenso de gradiente, cambia `eta` a 0.03 y a 0.001. ¿Qué ocurre con 0.03? ¿Y cuántos pasos harían falta con 0.001?

## 3. Una capa de red con NumPy

Se usan las 1000 primeras imágenes de MNIST, incluidas junto a este cuaderno.

> Datos: MNIST, de Y. LeCun, C. Cortes y C. J. C. Burges, con licencia Creative Commons Reconocimiento-CompartirIgual 3.0.

In [ ]:
import numpy as np

datos = np.load("mnist_muestra.npz")        # las 1000 primeras de MNIST
x_ent, y_ent = datos["x"], datos["y"]
lote = x_ent[:32]                         # un lote de 32 imágenes
print("lote:", lote.shape, lote.dtype)

X = lote.reshape(32, 784).astype(np.float32) / 255  # 32 vectores
print("imagen 0, píxeles 400 a 405:", X[0, 400:406].round(2))
print("imágenes que son un 7:", np.flatnonzero(y_ent[:32] == 7))

# una capa de red = producto de matrices + sesgo + activación
rng = np.random.default_rng(0)
W = rng.normal(0, 0.05, (784, 128)).astype(np.float32)
b = np.zeros(128, dtype=np.float32)
Z = X @ W + b                    # (32, 784) @ (784, 128) -> (32, 128)
H = np.maximum(Z, 0)             # ReLU: deja pasar solo lo positivo
print("salida de la capa:", H.shape,
      f"{(H > 0).mean():.0%} de las neuronas activas")

## 4. Una red neuronal es un grafo

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def red_en_capas(capas):
    """Grafo dirigido: cada neurona se une a todas las de la capa
    siguiente."""
    g = nx.DiGraph()
    for c, n in enumerate(capas):
        g.add_nodes_from(((c, i) for i in range(n)), capa=c)
    for c in range(len(capas) - 1):
        g.add_edges_from(((c, i), (c + 1, j))
                         for i in range(capas[c])
                         for j in range(capas[c + 1]))
    return g

g = red_en_capas([4, 5, 3])
print(g.number_of_nodes(), "neuronas,", g.number_of_edges(),
      "conexiones")
pos = nx.multipartite_layout(g, subset_key="capa")
nx.draw(g, pos, node_size=120, node_color="#1F5FA8",
        edge_color="#9AA5B1", arrowsize=6)
plt.savefig("grafo_red.pdf")

grande = red_en_capas([784, 128, 10])      # la red del apartado 1.10
pesos = grande.number_of_edges()
sesgos = 128 + 10                # uno por neurona oculta o de salida
print(f"red 784-128-10: {pesos:,} pesos + {sesgos} sesgos "
      f"= {pesos + sesgos:,} parámetros")
plt.show()

**Ejercicio.** Calcula con `red_en_capas` los parámetros de una red 784-256-128-10.

> Harris, C. R. et al. (2020). Array programming with NumPy. *Nature*, 585, 357–362. https://doi.org/10.1038/s41586-020-2649-2
>
> Virtanen, P. et al. (2020). SciPy 1.0. *Nature Methods*, 17, 261–272. https://doi.org/10.1038/s41592-019-0686-2